In [ ]:
import torch
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import urllib

In [ ]:
# 1. Load the pre-trained ResNet-18 model
# Note: weights=models.ResNet18_Weights.DEFAULT is the updated syntax for torchvision >= 0.13
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Set the model to evaluation mode (disables Dropout/BatchNorm updates)
model.eval()

In [ ]:
# 2. Download a sample image and preprocess it
url, filename = ("https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")
try:
    urllib.request.urlretrieve(url, filename)
except:
    pass

input_image = Image.open(filename)

# Define standard ImageNet preprocessing transforms
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])

input_tensor = preprocess(input_image)
input_batch = input_tensor.unsqueeze(0)  # Create a mini-batch of size 1

In [ ]:
# 3. Move to GPU if available
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
input_batch = input_batch.to(device)

In [ ]:
# 4. Perform Inference
with torch.no_grad():
    output = model(input_batch)

# Convert logits to probabilities
probabilities = torch.nn.functional.softmax(output[0], dim=0)

# Get top 5 predictions
top5_prob, top5_catid = torch.topk(probabilities, 5)
categories = weights.meta["categories"]

print("Top 5 Predictions:")
for i in range(top5_prob.size(0)):
    print(f"{categories[top5_catid[i]]}: {top5_prob[i].item() * 100:.2f}%")